In [1]:
# Setup environment
import os
os.chdir('/home/smallyan/relation_eval_agent')

# Set HF_HOME for cached models
os.environ['HF_HOME'] = '/net/projects2/chai-lab/shared_models'

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

CUDA available: True
CUDA device: NVIDIA A100 80GB PCIe


# Generalizability Evaluation for InterpDetect Repository

## Overview
This notebook evaluates the generalizability of findings in the InterpDetect repository using the Generalizability Checklist:
- **GT1**: Model Generalization - Test on new model (Pythia-1.4B)
- **GT2**: Data Generalization - Test on new data instances
- **GT3**: Method/Specificity Generalizability - Test method on another similar task

## Repository Location
`/net/scratch2/smallyan/InterpDetect_eval`

## Key Findings from Original Work:
1. **External Context Score (ECS)**: Cosine similarity between response and context embeddings - negatively correlated with hallucination
2. **Parametric Knowledge Score (PKS)**: Jensen-Shannon divergence in FFN layers - positively correlated with hallucination in later layers
3. **Original Model**: Qwen3-0.6B (28 layers, 16 attention heads)

In [2]:
# Load required libraries
import json
import numpy as np
from torch.nn import functional as F
import warnings
warnings.filterwarnings('ignore')

# Load test data
repo_path = '/net/scratch2/smallyan/InterpDetect_eval'
test_path = os.path.join(repo_path, 'datasets/test/test_w_chunk_score_qwen06b.json')
with open(test_path, 'r') as f:
    test_data = json.load(f)

print(f"Loaded {len(test_data)} test examples")

# Categorize examples
hallucinated_examples = []
truthful_examples = []

for ex in test_data:
    has_hallucination = any(s['hallucination_label'] == 1 for s in ex['scores'])
    if has_hallucination:
        hallucinated_examples.append(ex)
    else:
        truthful_examples.append(ex)

print(f"Examples with hallucinations: {len(hallucinated_examples)}")
print(f"Examples without hallucinations: {len(truthful_examples)}")

Loaded 256 test examples
Examples with hallucinations: 128
Examples without hallucinations: 128


---
## GT1: Model Generalization Test

**Objective**: Test if the ECS/PKS correlation findings generalize to a new model not used in the original work.

**Original Model**: Qwen3-0.6B (28 layers, 16 attention heads)

**New Model**: Pythia-1.4B (24 layers, 16 attention heads) - NOT used in original research

**Test**: We will compute ECS and PKS on Pythia-1.4B for a sample of hallucinated and truthful responses, and verify:
1. Later-layer PKS is higher for hallucinated responses
2. ECS is lower for hallucinated responses

In [3]:
# Load Pythia-1.4B model for GT1 evaluation
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer

print("Loading Pythia-1.4B model...")
pythia_model = HookedTransformer.from_pretrained(
    "EleutherAI/pythia-1.4b",
    device="cuda",
    dtype=torch.float16
)
pythia_tokenizer = AutoTokenizer.from_pretrained("EleutherAI/pythia-1.4b")

print(f"Pythia-1.4B loaded successfully")
print(f"Number of layers: {pythia_model.cfg.n_layers}")
print(f"Number of attention heads: {pythia_model.cfg.n_heads}")
print(f"Context length: {pythia_model.cfg.n_ctx}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Pythia-1.4B model...


Loaded pretrained model EleutherAI/pythia-1.4b into HookedTransformer


Pythia-1.4B loaded successfully
Number of layers: 24
Number of attention heads: 16
Context length: 2048


In [4]:
# Load BGE model for ECS computation
print("Loading BGE model...")
bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5").to("cuda")
print("BGE model loaded successfully")

Loading BGE model...


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [5]:
# Define helper functions for GT1 evaluation

def calculate_dist_2d(sep_vocabulary_dist, sep_attention_dist):
    """Calculate Jensen-Shannon divergence between distributions (PKS)"""
    softmax_mature_layer = F.softmax(sep_vocabulary_dist, dim=-1)
    softmax_anchor_layer = F.softmax(sep_attention_dist, dim=-1)
    M = 0.5 * (softmax_mature_layer + softmax_anchor_layer)
    log_softmax_mature_layer = F.log_softmax(sep_vocabulary_dist, dim=-1)
    log_softmax_anchor_layer = F.log_softmax(sep_attention_dist, dim=-1)
    kl1 = F.kl_div(log_softmax_mature_layer, M, reduction='none').sum(dim=-1)
    kl2 = F.kl_div(log_softmax_anchor_layer, M, reduction='none').sum(dim=-1)
    js_divs = 0.5 * (kl1 + kl2)
    scores = js_divs.cpu().tolist()
    return sum(scores)

def calculate_sentence_similarity(bge_model, r_text, p_text):
    """Calculate sentence similarity using BGE model (ECS)"""
    part_embedding = bge_model.encode([r_text], normalize_embeddings=True)
    q_embeddings = bge_model.encode([p_text], normalize_embeddings=True)
    scores_named = np.matmul(q_embeddings, part_embedding.T).flatten()
    return float(scores_named[0])

def compute_pks_for_pythia(model, input_ids, span_start, span_end):
    """Compute PKS scores across layers for Pythia model"""
    with torch.no_grad():
        logits, cache = model.run_with_cache(input_ids, return_type="logits")
    
    pks_scores = {}
    for layer_id in range(model.cfg.n_layers):
        x_mid = cache[f"blocks.{layer_id}.hook_resid_mid"][0, span_start:span_end, :]
        x_post = cache[f"blocks.{layer_id}.hook_resid_post"][0, span_start:span_end, :]
        score = calculate_dist_2d(x_mid @ model.W_U, x_post @ model.W_U)
        pks_scores[f"layer_{layer_id}"] = score
    
    return pks_scores, cache

def compute_ecs_for_pythia(model, cache, bge_model, prompt_spans, response_spans, 
                           original_prompt_spans, original_response_spans,
                           prompt, response, r_span_idx):
    """Compute ECS scores for a response span on Pythia"""
    r_span = response_spans[r_span_idx]
    
    ecs_scores = {}
    for layer_id in range(model.cfg.n_layers):
        for head_id in range(model.cfg.n_heads):
            attention_pattern = cache[f"blocks.{layer_id}.attn.hook_pattern"]
            attention_score = attention_pattern[0, head_id, :, :]
            
            # Find the prompt span with maximum attention
            p_span_scores = []
            for p_span in prompt_spans:
                attn_sum = torch.sum(attention_score[r_span[0]:r_span[1], p_span[0]:p_span[1]]).cpu().item()
                p_span_scores.append(attn_sum)
            
            # Get the span with maximum attention
            p_id = np.argmax(p_span_scores)
            prompt_span_text = prompt[original_prompt_spans[p_id][0]:original_prompt_spans[p_id][1]]
            respond_span_text = response[original_response_spans[r_span_idx][0]:original_response_spans[r_span_idx][1]]
            
            # Compute cosine similarity (ECS)
            ecs = calculate_sentence_similarity(bge_model, prompt_span_text, respond_span_text)
            ecs_scores[f"({layer_id}, {head_id})"] = ecs
    
    return ecs_scores

print("Helper functions defined")

In [6]:
print("Helper functions defined successfully")

In [7]:
# Force flush
import sys
sys.stdout.flush()
print("Helper functions defined", flush=True)

In [8]:
# Check if functions are defined
type(calculate_dist_2d)

In [9]:
# Test basic output
x = 1 + 1
x

In [10]:
raise ValueError("test output")